In [1]:
from models.base import ModelConfig
from models.llava_action import LLavaAction
from transformers import BitsAndBytesConfig

cfg = ModelConfig.from_dict({
    "model_name": "llava_qwen",
    "model_path": "MLAdaptiveIntelligence/LLaVAction-7B",  # or local path
    "max_new_tokens": 512,
    "temperature": 0,
    "device": "cuda",
    "device_map": {"": "cuda"},
    "dtype": "bfloat16",
    "quantization_config": BitsAndBytesConfig(load_in_8bit=True),
})

model = LLavaAction(cfg)


Loaded LLaVA model: MLAdaptiveIntelligence/LLaVAction-7B


You are using a model of type qwen2 to instantiate a model of type llava_qwen. This is not supported for all configurations of models and can yield errors.


Loading vision tower: google/siglip-so400m-patch14-384


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model Class: LlavaQwenForCausalLM


In [2]:
time_instruction = f"This is 1 video frame from from a surveillance footage."

perspective_prompt = "You are seeing this image from a surveillance camera view and you are the surveillance camera. What action is the person performing?"
task_prompt = "Describe in details what you see from the video frame."

base_text = f"\n{time_instruction}\n{perspective_prompt} {task_prompt}"

model.predict('night_guard_up.jpg', base_text)

/home/jeremy/code/synthetic_dataset_creation/synth_env/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'In the video frame, I see a dark, indistinct area that appears to be a floor or ground surface. There is no visible movement or interaction with objects, and no person is discernible in the frame. The lack of any visible action or object manipulation suggests that there is no action being performed by a person in this particular frame.'

In [ ]:
from PIL import Image

img_path = "dark_walking.jpg"
output_txt = model.predict(img_path)
print("Output:", output_txt)



RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_mm)

In [1]:
from llavaction.model.builder import load_pretrained_model
from llavaction.mm_utils import get_model_name_from_path, process_images, tokenizer_image_token
from llavaction.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN, IGNORE_INDEX
from llavaction.conversation import conv_templates, SeparatorStyle
from PIL import Image
import requests
import copy
import torch
import sys
import warnings
from decord import VideoReader, cpu
import numpy as np

pretrained = "MLAdaptiveIntelligence/LLaVAction-7B"
model_name = "llava_qwen"
device = "cuda"
device_map = "auto"
tokenizer, model, image_processor, max_length = load_pretrained_model(pretrained, None, model_name, torch_dtype="bfloat16", device_map=device_map)

Loaded LLaVA model: MLAdaptiveIntelligence/LLaVAction-7B


You are using a model of type qwen2 to instantiate a model of type llava_qwen. This is not supported for all configurations of models and can yield errors.


Loading vision tower: google/siglip-so400m-patch14-384


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

/home/jeremy/code/synthetic_dataset_creation/synth_env/lib/python3.11/site-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline', 'model.action_supervision']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.


Model Class: LlavaQwenForCausalLM
